In [1]:
%load_ext autoreload

In [ ]:
import contextlib
import logging
import os
import pickle
import torch
import numpy as np
import torch.nn.functional as F


import utils
import common
import sampler
import backbones
import patchcore
import metrics
from Multiview_dataset import MVTecMultiViewDataset, DatasetSplit

logging.basicConfig(level=logging.INFO)
LOGGER = logging.getLogger(__name__)


#load fn from 
class EarlyFusionPatchCore(patchcore.PatchCore):
    def __init__(self, device):
        super(EarlyFusionPatchCore, self).__init__(device)
        
    def modify_backbone_for_6channel(self):
        if hasattr(self.backbone, 'conv1'):
            original_conv = self.backbone.conv1
            original_weights = original_conv.weight.data
            new_conv = torch.nn.Conv2d(
                6,  
                original_conv.out_channels,
                kernel_size=original_conv.kernel_size,
                stride=original_conv.stride,
                padding=original_conv.padding,
                bias=original_conv.bias is not None
            ).to(self.device)
            new_weights = torch.zeros_like(new_conv.weight.data)
            new_weights[:, :3, :, :] = original_weights
            new_weights[:, 3:, :, :] = original_weights
            
            new_conv.weight.data = new_weights
            if original_conv.bias is not None:
                new_conv.bias.data = original_conv.bias.data
            
            self.backbone.conv1 = new_conv
        else:
            LOGGER.warning(
                "Could not modify backbone for 6-channel input. "
                "Backbone does not have 'conv1' attribute."
            )
            
    def load(
        self,
        backbone,
        layers_to_extract_from,
        device,
        input_shape,
        pretrain_embed_dimension,
        target_embed_dimension,
        patchsize=3,
        patchstride=1,
        anomaly_score_num_nn=1,
        featuresampler=patchcore.sampler.IdentitySampler(),
        nn_method=patchcore.common.FaissNN(False, 4),
        **kwargs,
    ):
    

        self.backbone = backbone.to(device)
        self.layers_to_extract_from = layers_to_extract_from
        self.input_shape = input_shape
        self.device = device
        self.patch_maker = patchcore.PatchMaker(patchsize, stride=patchstride)

        self.modify_backbone_for_6channel()
        

        self.forward_modules = torch.nn.ModuleDict({})

        feature_aggregator = patchcore.common.NetworkFeatureAggregator(
            self.backbone, self.layers_to_extract_from, self.device
        )

        feature_dimensions = feature_aggregator.feature_dimensions(input_shape)
        
        self.forward_modules["feature_aggregator"] = feature_aggregator

        preprocessing = patchcore.common.Preprocessing(
            feature_dimensions, pretrain_embed_dimension
        )
        self.forward_modules["preprocessing"] = preprocessing
        
        self.target_embed_dimension = target_embed_dimension

        preadapt_aggregator = patchcore.common.Aggregator(
            target_dim=target_embed_dimension
        )
        _ = preadapt_aggregator.to(self.device)
        self.forward_modules["preadapt_aggregator"] = preadapt_aggregator

        self.anomaly_scorer = patchcore.common.NearestNeighbourScorer(
            n_nearest_neighbours=anomaly_score_num_nn, nn_method=nn_method
        )

        self.anomaly_segmentor = patchcore.common.RescaleSegmentor(
            device=self.device, target_size=input_shape[-2:]
        )

        self.featuresampler = featuresampler
        
 
        LOGGER.info("Early fusion PatchCore initialized with 6-channel input.")

#mostly adapted from https://github.com/amazon-science/patchcore-inspection/blob/main/bin/run_patchcore.py
def run_patchcore_early_fusion(

    data_path,
    classnames,  
    batch_size=2,
    resize=256,
    imagesize=224,
    num_workers=8,
    
    backbone_name="wideresnet50",
    layers_to_extract_from=["layer2", "layer3"],
    pretrain_embed_dimension=1024,
    target_embed_dimension=1024,
    patchsize=3,
    patchstride=1,
    anomaly_scorer_num_nn=5,
    

    sampler_name="approx_greedy_coreset",
    percentage=0.1,
   
    gpu_id=0,
    seed=0,
    save_segmentation_images=False,
    save_patchcore_model=False,
    results_path="results",
    log_project="patchcore-project",
    log_group="patchcore-group"
):
    run_save_path = utils.create_storage_folder(
        results_path, log_project, log_group, mode="iterate"
    )
    

    device = torch.device(f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu")
    LOGGER.info(f"Using device: {device}")
 
    device_context = (
        torch.cuda.device(f"cuda:{gpu_id}")
        if "cuda" in device.type.lower()
        else contextlib.suppress()
    )

    utils.fix_seeds(seed, device)

    train_dataset = MVTecMultiViewDataset(
        data_path,
        classnames=classnames,
        resize=resize,
        imagesize=imagesize,
        split=DatasetSplit.TRAIN,
    )
    test_dataset = MVTecMultiViewDataset(
        data_path,
        classnames=classnames,
        resize=resize,
        imagesize=imagesize,
        split=DatasetSplit.TEST,
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    test_dataloader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    train_dataloader.name = f"mvtec_multiview_{classnames[0]}_{classnames[1]}"
    
    dataloaders = {
        "training": train_dataloader,
        "testing": test_dataloader,
    }

    if sampler_name == "identity":
        feature_sampler = sampler.IdentitySampler()
    elif sampler_name == "greedy_coreset":
        feature_sampler = sampler.GreedyCoresetSampler(percentage, device)
    elif sampler_name == "approx_greedy_coreset":
        feature_sampler = sampler.ApproximateGreedyCoresetSampler(percentage, device)
    elif sampler_name == "KNN_coreset":
        feature_sampler = sampler.FastKNNSampler(percentage, device)

    with device_context:
        torch.cuda.empty_cache()

        backbone = backbones.load(backbone_name)
        backbone.name, backbone.seed = backbone_name, None

        nn_method = common.FaissNN(True, 8) 

        patchcore_instance = EarlyFusionPatchCore(device)
        
        patchcore_instance.load(
            backbone=backbone,
            layers_to_extract_from=layers_to_extract_from,
            device=device,
            input_shape=(6, imagesize, imagesize),  
            pretrain_embed_dimension=pretrain_embed_dimension,
            target_embed_dimension=target_embed_dimension,
            patchsize=patchsize,
            patchstride=patchstride,
            anomaly_score_num_nn=anomaly_scorer_num_nn,
            featuresampler=feature_sampler,
            nn_method=nn_method,
        )
        

        LOGGER.info("Training EarlyFusion PatchCore model...")
        patchcore_instance.fit(dataloaders["training"])
 
        LOGGER.info("Testing EarlyFusion PatchCore model...")
        scores, segmentations, labels_gt, masks_gt = patchcore_instance.predict(
            dataloaders["testing"]
        )

        anomaly_labels = [label == 1 for label in labels_gt]
        unique_labels = set(labels_gt)
        print(f"Unique labels in labels_gt: {unique_labels}")
        label_counts = {label: labels_gt.count(label) for label in unique_labels}
        print(f"Label counts: {label_counts}")

        anomaly_labels = [label == 1 for label in labels_gt]
        has_normal = False in anomaly_labels
        has_anomaly = True in anomaly_labels
        print(f"Do we have normal samples? {has_normal}")
        print(f"Do we have anomalous samples? {has_anomaly}")

        normal_count = anomaly_labels.count(False)
        anomaly_count = anomaly_labels.count(True)
        print(f"Normal count: {normal_count}, Anomaly count: {anomaly_count}")

        if has_normal and has_anomaly:
            instance_auroc = metrics.compute_imagewise_retrieval_metrics(
                scores, anomaly_labels
            )["auroc"]
        else:
            print("WARNING: Only one class in test set! AUROC calculation is not possible.")
            instance_auroc = float('nan') 
  
        instance_auroc = metrics.compute_imagewise_retrieval_metrics(
            scores, anomaly_labels
        )["auroc"]
        
        # Pixel-level AUROC (full)
        # full_pixel_auroc = metrics.compute_pixelwise_retrieval_metrics(
        #     segmentations, masks_gt
        # )["auroc"]
        
        # Pixel-level AUROC (anomalies only)
        sel_idxs = []
        for i in range(len(masks_gt)):
            if np.sum(masks_gt[i]) > 0:
                sel_idxs.append(i)
                
        # anomaly_pixel_auroc = 0
        # if len(sel_idxs) > 0:
        #     anomaly_pixel_auroc = metrics.compute_pixelwise_retrieval_metrics(
        #         [segmentations[i] for i in sel_idxs],
        #         [masks_gt[i] for i in sel_idxs],
        #     )["auroc"]
        
        # Print results
        results = {
            "dataset_name": f"{classnames[0]}_{classnames[1]}",
            "instance_auroc": instance_auroc,
            #"full_pixel_auroc": full_pixel_auroc,
           # "anomaly_pixel_auroc": anomaly_pixel_auroc,
        }
        
        for key, item in results.items():
            if key != "dataset_name":
                LOGGER.info("{0}: {1:3.3f}".format(key, item))
        
        # Save model if requested
        if save_patchcore_model:
            patchcore_save_path = os.path.join(
                run_save_path, "models", f"{classnames[0]}_{classnames[1]}"
            )
            os.makedirs(patchcore_save_path, exist_ok=True)
            patchcore_instance.save_to_path(patchcore_save_path)
        
        return results



In [ ]:
results = run_patchcore_early_fusion(
    data_path=r"C:\Thesis\Data\paperclips_sorted",
    classnames=["paperclips_front", "paperclips_side"])

INFO:__main__:Using device: cuda:0
c:\Users\feder\anaconda3\envs\patchcore-env\lib\site-packages\torch\utils\data\dataloader.py:557: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 6 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
c:\Users\feder\anaconda3\envs\patchcore-env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\feder\anaconda3\envs\patchcore-env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 

Unique labels in labels_gt: {0, 1}
Label counts: {0: 44, 1: 54}
Do we have normal samples? True
Do we have anomalous samples? True
Normal count: 44, Anomaly count: 54


INFO:__main__:instance_auroc: 0.633
